# Mission 3 TF-IDF + Logistic Regression baseline

이 노트북은 저장소의 현재 `scripts/mission3_baseline.py`와 `scripts/inference.py`를 기준으로 실행합니다.

## 공식 규정 확인 기준

- K-ICT 빅데이터센터 공식 메일 **`[2026 DCC] 대학부 규정 안내` (2026-09-21)**: 미션별 `inference.py`를 한 번 실행해 결과 파일을 생성할 수 있어야 하며, `requirements.txt`로 실행 환경을 재현할 수 있어야 합니다.
- 같은 공식 메일의 Decision Threshold 규정: **모든 class에 동일하게 0.5를 고정**하며, Train/Validation/OOF를 이용한 threshold tuning도 허용되지 않습니다.
- Mission 3 출제 규칙에 따라 모델 입력은 **대화 전사 텍스트**만 사용합니다. `startAt`, `id`, `recordId` 등 다른 annotation은 모델 입력·정렬·분할에 사용하지 않습니다.
- 학습 정답은 Training JSON의 `symptom` 중 평가 대상 9개만 사용합니다.
- 내부 검증은 Training을 80:20 multilabel stratified split, seed=42로 나눕니다.
- 공식 Validation은 학습·threshold 조정에 사용하지 않고 최종 추론 확인에만 사용합니다.


In [ ]:
# Colab에서 최신 저장소를 준비합니다.
# 이미 /content/prelim이 있다면 다시 clone하지 않습니다.
from pathlib import Path
import os

REPO_DIR = Path('/content/prelim')

if not REPO_DIR.exists():
    !git clone https://github.com/eigensinn-dcc2026/prelim.git /content/prelim

%cd /content/prelim
!python -m pip install -q -r requirements.txt


## 데이터 경로 설정

개인 Google Drive 경로를 노트북에 고정하지 않습니다. `DCC_DATA_DIR`에는 주최 측이 제공한 대학부 데이터의 **로컬 루트 폴더**를 지정합니다.


In [ ]:
from pathlib import Path
import os

# Colab/로컬 환경에서 공식 데이터가 있는 루트만 지정합니다.
# 특정 팀원의 개인 저장경로를 노트북에 하드코딩하지 않습니다.
DATA_DIR = Path(os.environ.get('DCC_DATA_DIR', '/content/data')).expanduser()

TRAIN_LABEL_DIR = DATA_DIR / 'Training' / '2.라벨링데이터' / 'TL_서울_구급'
VALID_LABEL_DIR = DATA_DIR / 'Validation' / '2.라벨링데이터' / 'VL_서울_구급'
VALID_AUDIO_DIR = DATA_DIR / 'Validation' / '1.원천데이터' / 'VS_서울_구급'

print('DATA_DIR:', DATA_DIR)
print('Training JSON 폴더:', TRAIN_LABEL_DIR, TRAIN_LABEL_DIR.exists())
print('Validation JSON 폴더:', VALID_LABEL_DIR, VALID_LABEL_DIR.exists())
print('Validation WAV 폴더:', VALID_AUDIO_DIR, VALID_AUDIO_DIR.exists())

assert TRAIN_LABEL_DIR.exists(), 'Training JSON 폴더를 찾지 못했습니다. DCC_DATA_DIR를 확인하세요.'


In [ ]:
# 공식 안내의 데이터 개수와 현재 경로가 맞는지 먼저 확인합니다.
train_json_files = sorted(TRAIN_LABEL_DIR.rglob('*.json'))
valid_json_files = sorted(VALID_LABEL_DIR.rglob('*.json'))

print(f'Training JSON: {len(train_json_files):,}개 / 공식 안내 {EXPECTED_TRAIN_COUNT:,}개')
print(f'Validation JSON: {len(valid_json_files):,}개 / 공식 안내 {EXPECTED_VALID_COUNT:,}개')

if len(train_json_files) != EXPECTED_TRAIN_COUNT:
    print('⚠️ Training 개수가 공식 안내와 다릅니다. 일부 하위 폴더만 선택하지 않았는지 확인하세요.')
if len(valid_json_files) != EXPECTED_VALID_COUNT:
    print('⚠️ Validation 개수가 공식 안내와 다릅니다. 데이터 루트 경로를 확인하세요.')

assert train_json_files, 'Training JSON을 찾지 못했습니다.'


In [ ]:
# 샘플 JSON의 M3 관련 항목만 확인합니다.
import json

with train_json_files[0].open('r', encoding='utf-8-sig') as file:
    sample_data = json.load(file)

print('symptom:', sample_data.get('symptom'))
print('발화 개수:', len(sample_data.get('utterances', [])))
print('첫 번째 발화 text:', sample_data.get('utterances', [])[0].get('text', ''))


In [ ]:
# 현재 main의 baseline 함수와 동일한 이름을 사용합니다.
from scripts.mission3_baseline import load_training_dataframe, make_dataset_summary

dataframe = load_training_dataframe(TRAIN_LABEL_DIR)
summary = make_dataset_summary(dataframe)

print('불러온 데이터 개수:', len(dataframe))
print('빈 전사 수:', summary['empty_text_count'])
print('목표 9개 증상이 하나도 없는 샘플 수:', summary['zero_target_label_count'])
display(dataframe.head())


In [ ]:
# 학습에 실제로 쓰이는 9개 증상과 그 외 symptom을 확인합니다.
display(
    dataframe[['label_file_name', 'symptoms', 'non_target_symptoms']].head(10)
)


In [ ]:
# Training JSON만 이용해 TF-IDF + Logistic Regression baseline을 학습합니다.
# 공식 규정상 Decision Threshold는 0.5로 고정이며 조정하지 않습니다.
train_label_path = str(TRAIN_LABEL_DIR)

!python scripts/mission3_baseline.py \
  --label_dir "$train_label_path" \
  --output_dir "./artifacts" \
  --threshold 0.5 \
  --test_size 0.2 \
  --seed 42


In [ ]:
# 학습 결과가 공식 threshold 규칙을 지켰는지 확인합니다.
import json

with open('./artifacts/metrics.json', 'r', encoding='utf-8') as file:
    metrics = json.load(file)

print('Validation Macro F1:', metrics['macro_f1'])
print('Decision Threshold:', metrics['threshold'])
assert metrics['threshold'] == 0.5, '공식 규정상 threshold는 반드시 0.5여야 합니다.'


In [ ]:
# 공식 Validation에 대한 최종 예측 파일을 만듭니다.
# inference.py는 JSON에서 utterances[].text만 읽고 symptom 정답은 사용하지 않습니다.
validation_label_path = str(VALID_LABEL_DIR)
validation_audio_path = str(VALID_AUDIO_DIR)

assert VALID_LABEL_DIR.exists(), 'Validation JSON 폴더를 찾지 못했습니다.'
assert VALID_AUDIO_DIR.exists(), 'Validation WAV 폴더를 찾지 못했습니다.'

!python scripts/inference.py \
  --audio_dir "$validation_audio_path" \
  --label_dir "$validation_label_path" \
  --ckpt_path "./artifacts/mission3_tfidf_logreg.joblib" \
  --output "./outputs/mission3.csv"


In [ ]:
# 생성된 Mission 3 결과 형식을 확인합니다.
import pandas as pd

submission = pd.read_csv('./outputs/mission3.csv')
validation_json_count = len(list(VALID_LABEL_DIR.rglob('*.json')))

print('예측 행 수:', len(submission))
print('Validation JSON 수:', validation_json_count)
display(submission.head())

assert list(submission.columns) == ['label file name', 'symptom'], 'Mission 3 CSV 열 이름을 확인하세요.'
assert len(submission) == validation_json_count, '입력 JSON 수와 출력 CSV 행 수가 다릅니다.'


## 제출 전 주의

이 PR의 자동 검사는 코드 구조와 기본 테스트를 확인하는 것이며, 실제 대회 데이터 실행을 대신하지 않습니다. 최종 제출용 학습 노트북은 **공식 Training 데이터로 실제 실행한 학습 로그와 주요 결과가 남아 있는 상태**로 저장하고, `inference.py` 단일 실행으로 결과 CSV가 생성되는지 다시 확인합니다.
